# 01 · Bronze ingest
Auto Loader lands all 9 CSVs as STRING plus audit columns (module 05). Incremental and idempotent: a re-run with no new files adds 0 rows.

✅ Expect **26,168** rows in total.

In [ ]:
from pyspark.sql import functions as F
from hcm_utils import load_config, resolve_paths

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("run_id", "manual")
catalog = dbutils.widgets.get("catalog")
run_id  = dbutils.widgets.get("run_id")
cfg = load_config()
paths = resolve_paths(cfg, catalog)

landing = paths["landing"]
ckpt    = paths["checkpoints"]

In [ ]:
def ingest(table: str):
    src = f"{landing}/{table}/"
    q = (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.schemaLocation", f"{ckpt}/{table}/_schema")
            .load(src)
            .select("*", F.col("_metadata.file_path").alias("SourceFileName"))
            .withColumn("BronzeLoadTimestamp", F.current_timestamp())
            .withColumn("SourceSystem", F.lit(cfg["source_system"]))
            .withColumn("RunId", F.lit(run_id))
            .withColumn("SchemaVersion", F.lit(cfg["schema_version"]))
         .writeStream
            .option("checkpointLocation", f"{ckpt}/{table}/_checkpoint")
            .option("mergeSchema", "true")
            .trigger(availableNow=True)
            .toTable(f"{catalog}.bronze.{table}"))
    q.awaitTermination()

for t in cfg["bronze_tables"]:
    ingest(t)
    print(f"{t}: {spark.table(f'{catalog}.bronze.{t}').count():,} rows")

In [ ]:
total = sum(spark.table(f"{catalog}.bronze.{t}").count() for t in cfg["bronze_tables"])
dbutils.jobs.taskValues.set(key="bronze_total_rows", value=total)
print(f"TOTAL bronze rows: {total:,}")